# Experiment 2: Geometric Consistency

**Criterion 2**: Per-layer Spearman ρ > 0.5, mean > 0.65

Measures whether z-space distances faithfully reflect backbone-space distances at each layer.
Includes per-layer UMAP visualization of z-space colored by class.

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.metrics import geometric_consistency
from evaluation.circuit_viz import plot_per_layer_umap

In [ ]:
# Load config and override data directory
config_path     = CONFIG_DIR + '/phase1.yaml'
checkpoint_path = CKPT_DIR  + '/phase1/best.pt'

with open(config_path) as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = DATA_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Build models and load weights
backbone = FrozenBackbone(
    arch=config['model']['arch'],
    num_classes=config['model']['num_classes'],
    pretrained=config['model']['pretrained'],
).to(device)

meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    projection_dim=config['model']['meta_encoder']['projection_dim'],
    n_heads=config['model']['meta_encoder']['n_heads'],
    n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
    dropout=config['model']['meta_encoder']['dropout']
).to(device)

ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
meta_encoder.eval()
print('Model loaded.')

In [ ]:
# Create validation loader, collect representations, compute pair profiles
_, val_loader = get_standard_loaders(
    data_dir=DATA_DIR,
    batch_size=config['data'].get('batch_size', 256),
    num_workers=0,
    augment=False,
    download=False,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
data     = analyzer.collect_representations(max_samples=2000)
print(f'Collected {data["labels"].shape[0]} samples')

# Pair indices (subsample to 50k)
N = data['z_list'][0].shape[0]
idx_a, idx_b = torch.triu_indices(N, N, offset=1)
MAX_PAIRS = 50_000
if idx_a.shape[0] > MAX_PAIRS:
    perm  = torch.randperm(idx_a.shape[0])[:MAX_PAIRS]
    idx_a, idx_b = idx_a[perm], idx_b[perm]

# True pair profiles [N_pairs, L]
true_sims    = CircuitAnalyzer.compute_pair_profiles(data['trajectories'], idx_a, idx_b)
true_sims_np = true_sims.numpy()

# z-space pair cosine similarities [N_pairs, L]
L = len(data['z_list'])
z_sims_np = np.zeros((idx_a.shape[0], L))
for l in range(L):
    z_sims_np[:, l] = (
        data['z_list'][l][idx_a].numpy() * data['z_list'][l][idx_b].numpy()
    ).sum(axis=1)

print(f'N pairs: {idx_a.shape[0]}, L layers: {L}')

In [ ]:
# Criterion 2: Geometric Consistency
result = geometric_consistency(z_sims_np, true_sims_np, L)
print(f"Mean Spearman \u03c1: {result['mean_rho']:.4f}  "
      f"(target > 0.65, {'PASS' if result['passes'] else 'FAIL'})")
for l, rho in enumerate(result['per_layer_rho']):
    status = '\u2713' if rho > 0.5 else '\u2717'
    print(f'  Layer {l+1}: \u03c1 = {rho:.4f}  {status}')

In [ ]:
# Per-layer Spearman rho bar chart
rhos   = result['per_layer_rho']
layers = [f'L{l+1}' for l in range(L)]
colors = ['steelblue' if r > 0.5 else 'coral' for r in rhos]

fig, ax = plt.subplots(figsize=(max(6, L), 4))
ax.bar(layers, rhos, color=colors)
ax.axhline(0.5,  color='r', linestyle='--', alpha=0.7, label='Per-layer target')
ax.axhline(result['mean_rho'], color='navy', linestyle=':', lw=2, label='Mean')
ax.set_ylabel('Spearman \u03c1')
ax.set_title('Geometric Consistency per Layer')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-layer UMAP visualization (uses z_list directly)
z_np     = [z.numpy() for z in data['z_list']]
labels_np = data['labels'].numpy()

fig = plot_per_layer_umap(z_np, labels_np)
plt.show()